# PI-MADRL black start — training notebook (IEEE 14 / 39 / 118)

Runs the full experiment matrix on your own hardware and writes every JSON the manuscript
tables and figures are generated from.

**Every cell is resumable.** Each finished training run is written to `<out>/runs/` immediately;
re-running a cell after an interruption skips what is already there and continues. You can stop
the kernel at any point and pick up later.

**Order matters.** Run the cells top to bottom the first time. `baselines` must finish before the
learned results are interpretable, because the relaxed bound is what the LRR is judged against.

**Expected wall-clock** (single core; divide by `JOBS` if you set it higher):

| block | runs | 14-bus | 39-bus | 118-bus |
|---|---|---|---|---|
| baselines | – | ~2 min | ~5 min | ~25 min |
| benchmark (7 algos × 3 seeds) | 21 | ~30 min | ~2 h | ~5 h |
| fair (6 new × 3 seeds) | 18 | ~30 min | ~2 h | ~5 h |
| ablation (8 new × 3 seeds) | 24 | ~35 min | – | – |
| scalability (118 only) | – | – | – | ~30 min |

Set `JOBS` to the number of physical cores you can spare — the matrix parallelises across runs
almost linearly.

## 1. Environment and packages

In [1]:
# Install everything the simulator needs. Safe to re-run; pip skips what is satisfied.
# On a fresh conda env:  conda create -n blackstart python=3.11 -y && conda activate blackstart
import sys, subprocess

PKGS = [
    "numpy>=1.26",
    "scipy>=1.11",
    "torch>=2.1",        # CPU build is enough; the environment, not the network, dominates
    "pandapower>=3.0",   # test-case data + the independent power-flow cross-check
    "matplotlib>=3.8",
    "pyyaml>=6.0",
    "pytest>=7.4",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PKGS])
print("packages ready")

packages ready


In [2]:
# Point REPO at the folder that contains `sim/`, `configs/`, `figs/`.
# If this notebook sits in <repo>/notebooks/ the default is already correct.
import os, sys, json, time, pathlib

REPO = str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == "notebooks" else str(pathlib.Path.cwd())
sys.path.insert(0, REPO)
os.chdir(REPO)

OUT   = os.path.join(REPO, "sim", "results", "final")   # everything is written here
JOBS  = max(1, (os.cpu_count() or 2) - 1)               # parallel training runs
SEEDS = 3                                               # 3 for the reported matrix; 10 resolves the top-3 ranking
EPISODES = 50                                          # raise to 1200-3000 if you have the compute1

os.makedirs(OUT, exist_ok=True)
import torch; torch.set_num_threads(1)   # one thread per run; JOBS gives the parallelism
print(f"repo   = {REPO}")
print(f"out    = {OUT}")
print(f"jobs   = {JOBS} | seeds = {SEEDS} | episodes = {EPISODES}")

repo   = C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2
out    = C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final
jobs   = 15 | seeds = 3 | episodes = 50


In [3]:
# Sanity check: 31 tests must pass before any result is trustworthy (~2 min).
!python -m pytest -q

...............................                                          [100%]
============================== warnings summary ===============================
tests/test_algos.py: 142 warnings
tests/test_env.py: 26 warnings
tests/test_physics.py: 15 warnings
  C:\ProgramData\anaconda3\envs\blackstart\Lib\site-packages\pandapower\build_branch.py:525: DeprecationWarning: tap_dependency_table is missing in net, which is most probably due to unsupported net data. tap_dependency_table was introduced with pandapower 3.0 and replaced spline characteristics. Spline characteristics will still work, but they are deprecated and will be removed in future releases.
    warnings.warn(DeprecationWarning("tap_dependency_table is missing in net, which is most probably due to "

tests/test_algos.py: 142 warnings
tests/test_env.py: 26 warnings
tests/test_physics.py: 15 warnings
  C:\ProgramData\anaconda3\envs\blackstart\Lib\site-packages\pandapower\build_branch.py:1087: DeprecationWarning: tap_dependenc

In [4]:
# The three test systems exactly as the code builds them.
# Copy this output into Table 2 of the manuscript -- do not retype the ratings.
!python -m sim.networks

{"system": "case14", "buses": 14, "branches": 20, "lines": 15, "trafos": 5, "switchable": 20, "loads": 11, "load_mw": 259.0, "load_mvar": 73.5, "charging_mvar": 22.72, "pv": [100.0], "wp": [110.0], "ess_mw": [75.0], "ess_mwh": [150.0], "ibr_mw": 285.0, "ibr_mva": 316.667, "adequacy": 1.1, "notes": {"surrogate_ratings": 20, "taps_reset": [0.9780000000000001, 0.9690000000000001, 0.932]}}
{"system": "case39", "buses": 39, "branches": 46, "lines": 35, "trafos": 11, "switchable": 20, "loads": 28, "load_mw": 1035.7, "load_mvar": 229.704, "charging_mvar": 310.839, "pv": [110.0, 110.0, 110.0, 110.0], "wp": [100.0, 100.0, 100.0, 100.0], "ess_mw": [125.0, 100.0, 100.0], "ess_mwh": [250.0, 200.0, 200.0], "ibr_mw": 1165.0, "ibr_mva": 1294.444, "adequacy": 1.125, "notes": {"split_loads": [20, 10, 4, 2, 8, 1, 7], "taps_reset": [1.025, 1.07, 1.07, 1.006, 1.006, 1.06, 1.07, 1.009, 1.025, 1.025, 1.025], "shunts_out_mvar": 0.0}}
{"system": "case118", "buses": 118, "branches": 186, "lines": 173, "trafos"

In [5]:
# Parameters still marked PROVISIONAL. Every one of these has to be confirmed or replaced
# before Table 1 can claim reproducibility.
!python -m sim.config --provisional

limits.v_trip                          [0.85, 1.15]                 [PROVISIONAL: under/over-voltage protection band -> island collapse]
limits.df_trip_hz                      1.5                          [PROVISIONAL: under/over-frequency protection -> island collapse]
limits.sync_df_hz                      0.1                          [standard:IEEE 1547-2018 Table 5, DER > 1500 kVA; reproduced in Fix et al., ISGT 2023, Table II]
limits.sync_dv_pu                      0.03                         [standard:IEEE 1547-2018 Table 5, DER > 1500 kVA; reproduced in Fix et al., ISGT 2023, Table II]
limits.sync_dtheta_deg                 10.0                         [standard:IEEE 1547-2018 Table 5, DER > 1500 kVA; reproduced in Fix et al., ISGT 2023, Table II]
gfm.t_lpf_s                            0.10                         [PROVISIONAL: droop power-measurement filter; makes the response 2nd order (nadir below settling value)]
gfm.pf_rated                           0.9                   

## 2. Reference policies and the feasibility bound

Run this **before** the learned results. A load recovery ratio means nothing without the masked-random
policy, the greedy operator rule and the relaxed bound beside it: on these systems a masked random
policy already reaches 0.74 / 0.77 / 0.42, so anything near those numbers is not evidence of learning.

The 118-bus uses the checkpointing runner, which saves after every scenario — interrupt it freely.

In [6]:
# 14- and 39-bus reference policies (+ beam search on the 14-bus).
!python -m sim.run_experiments --systems 14 39 --parts baselines --out {OUT} \
    --label "final: 75-min window, w_f=2.0"

wrote C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final\meta.json
[baselines] 14: random+mask LRR=0.729 greedy LRR=0.791 bound=0.951
[baselines] 39: random+mask LRR=0.781 greedy LRR=0.794 bound=0.984
wrote C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final\baselines.json
wrote C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final\meta.json
ALL DONE in 3.1 min -> C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final


fatal: not a git repository (or any of the parent directories): .git


In [7]:
# 118-bus reference policies. Resumable: re-run this cell until it prints DONE.
from sim.baselines_incremental import run as run_baselines_118

while run_baselines_118("118", OUT, budget_s=600.0) is None:
    print("  ... continuing", flush=True)

[118] random+mask: complete, LRR=0.415
[118] greedy+mask: complete, LRR=0.816
[118] DONE  random+mask=0.415 greedy=0.816 bound=1.000


## 3. Benchmark — seven algorithms per system

DQN, IQL, VDN, QMIX, PPO, MAPPO and PI-MAPPO, each with the same number of environment steps.
One cell per system so you can run them on different machines or over different days.

In [12]:
!python -m sim.run_experiments --systems 14 --parts benchmark \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

wrote C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final\meta.json

fatal: not a git repository (or any of the parent directories): .git


[benchmark   1/21]  14/dqn               s0 ret= -4.44 LRR=0.071 VVR= 0.0% FVR=12.9% fail= 14% (219s, total 0.1 min)


[benchmark   2/21]  14/dqn               s1 ret= -5.56 LRR=0.037 VVR= 0.0% FVR=17.9% fail=  2% (218s, total 0.1 min)
[benchmark   3/21]  14/dqn               s2 ret= -3.29 LRR=0.084 VVR= 0.0% FVR= 1.1% fail=  0% (222s, total 0.1 min)
[benchmark   4/21]  14/iql               s1 ret= -0.31 LRR=0.270 VVR= 0.0% FVR= 9.3% fail=  6% (284s, total 0.1 min)
[benchmark   5/21]  14/iql               s0 ret= -0.57 LRR=0.301 VVR= 0.0% FVR= 6.5% fail=  0% (276s, total 0.1 min)
[benchmark   6/21]  14/iql               s2 ret= +0.35 LRR=0.269 VVR= 0.0% FVR= 6.9% fail=  2% (289s, total 0.1 min)
[benchmark   7/21]  14/vdn               s0 ret= -2.41 LRR=0.093 VVR= 3.4% FVR= 8.4% fail= 20% (288s, total 0.1 min)
[benchmark   8/21]  14/qmix              s0 ret= -0.34 LRR=0.267 VVR= 0.0% FVR= 4.4% fail=  0% (319s, total 0.1 min)
[benchmark   9/21]  14/vdn               s2 ret= -0.77 LRR=0.255 VVR= 0.0% FVR= 3.7% fail=  0% (285s, total 0.1 min)
[benchmark  10/21]  14/vdn               s1 ret= -0.36 LRR=0.31

In [13]:
!python -m sim.run_experiments --systems 39 --parts benchmark \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

fatal: not a git repository (or any of the parent directories): .git

wrote C:\Users\mikram0\Documents\MADRL_BlackStart_v2.7\MADRL_BlackStart_v2\sim\results\final\meta.json


[benchmark   1/21]  39/dqn               s0 ret= -2.33 LRR=0.081 VVR= 0.0% FVR= 1.4% fail=  0% (758s, total 0.1 min)
[benchmark   2/21]  39/dqn               s1 ret= -5.21 LRR=0.029 VVR= 8.2% FVR=15.6% fail= 74% (755s, total 0.1 min)
[benchmark   3/21]  39/dqn               s2 ret= -3.63 LRR=0.001 VVR= 0.0% FVR= 0.0% fail=  0% (790s, total 0.1 min)
[benchmark   4/21]  39/iql               s0 ret= -2.24 LRR=0.013 VVR=10.8% FVR=27.4% fail= 82% (866s, total 0.1 min)
[benchmark   5/21]  39/iql               s1 ret= -1.83 LRR=0.042 VVR= 0.0% FVR= 0.2% fail=  0% (859s, total 0.1 min)
[benchmark   6/21]  39/iql               s2 ret= -3.08 LRR=0.135 VVR= 1.1% FVR=18.2% fail= 10% (879s, total 0.1 min)
[benchmark   7/21]  39/vdn               s0 ret= -3.74 LRR=0.039 VVR= 0.0% FVR= 2.3% fail=  0% (962s, total 0.1 min)
[benchmark   8/21]  39/vdn               s1 ret= -3.72 LRR=0.000 VVR=26.1% FVR=100.0% fail=100% (949s, total 0.1 min)
[benchmark   9/21]  39/vdn               s2 ret= -4.00 LRR=0.00

In [ ]:
!python -m sim.run_experiments --systems 118 --parts benchmark \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

## 4. Fair comparison — every algorithm behind the same interface

This is the table that isolates the learner from the physics-informed interface. Each algorithm gets
the mask, the shield and the $\bm{\phi}^{\mathrm{PI}}$ features; only the learning rule differs.

In [ ]:
!python -m sim.run_experiments --systems 14 --parts fair \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

In [ ]:
!python -m sim.run_experiments --systems 39 --parts fair \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

In [ ]:
!python -m sim.run_experiments --systems 118 --parts fair \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

## 5. Component ablation (14-bus) and scalability stress test (118-bus)

The ablation tests each of the five physics-informed components in both directions: added alone to
MAPPO, and removed alone from PI-MAPPO. The stress test adds 0/18/36/54/72 priority loads to the
118-bus system and re-evaluates the trained policy without retraining.

In [ ]:
!python -m sim.run_experiments --systems 14 --parts ablation \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

In [ ]:
!python -m sim.run_experiments --systems 118 --parts scalability \
    --seeds {SEEDS} --episodes {EPISODES} --jobs {JOBS} --out {OUT}

## 6. Progress check

Run any time to see how far the matrix has got.

In [ ]:
import collections, glob

EXPECTED = {("benchmark", s): 7 * SEEDS for s in ("14", "39", "118")}
EXPECTED.update({("fair", s): 7 * SEEDS for s in ("14", "39", "118")})
EXPECTED[("ablation", "14")] = 10 * SEEDS

have = collections.Counter()
for f in glob.glob(os.path.join(OUT, "runs", "*.json")):
    part, system = os.path.basename(f).split("_")[:2]
    have[(part, system)] += 1

for key in sorted(EXPECTED):
    n, tot = have.get(key, 0), EXPECTED[key]
    bar = "#" * int(24 * n / tot) + "." * (24 - int(24 * n / tot))
    print(f"{key[0]:<10} {key[1]:>4}-bus  [{bar}] {n:>3}/{tot}")

for f in ("baselines.json", "benchmark.json", "fair.json", "ablation.json", "scalability.json"):
    p = os.path.join(OUT, f)
    if os.path.exists(p):
        with open(p) as fh:
            d = json.load(fh)
        print(f"{f:<20} systems: {[k for k in d if not k.startswith('_')]}")
    else:
        print(f"{f:<20} not written yet")

## 7. Tables, claims and figures

Run this only once the matrix is complete. `claims.md` contains every number the manuscript text
quotes; the `.tex` tables are `\input` directly by `main.tex`.

In [ ]:
!python -m sim.make_tables --results {OUT}
!python figs/make_comparison_figures.py --results {OUT}
print(open(os.path.join(OUT, "claims.md")).read())

## 8. Package the results

Send the archive this writes; it contains every JSON, table and figure needed to write the paper.

In [ ]:
import shutil

archive = shutil.make_archive(os.path.join(REPO, "results_final"), "zip", OUT)
print("wrote", archive, f"({os.path.getsize(archive) / 1e6:.1f} MB)")